# Performant visualizations of large catalogs in Aladin


In [ ]:
import io
import os
import requests
from zipfile import ZipFile
from pathlib import Path

import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.vizier import Vizier

data_dir = Path("performance_catalog_data")
download_catalogs = True

if download_catalogs:
    url = "https://stsci.box.com/shared/static/f20hg260eideeoxaneiyf99zu70fpijr.zip"  # [8 MB]
    r = requests.get(url)
    z = ZipFile(io.BytesIO(r.content))
    z.extractall()

!ls {data_dir}

<br />
To reproduce the files in the ZIP archive, open the source code below:

<details>
    <summary>source code</summary>

```
catalog = Vizier(
    catalog='IV/39/tic82',
    row_limit=100_000
)

result = catalog.query_object("omega Cen", radius=0.2 * u.deg)[0]
result.write(data_dir + 'omega_Cen_TIC.ecsv')

catalog = Vizier(
    catalog='J/ApJS/271/26',
    row_limit=100_000
)

result = catalog.query_object("omega Cen", radius=6 * u.deg)[0]
result.write(data_dir + 'omega_Cen_galex.ecsv', overwrite=True)

catalog = Vizier(
    catalog='I/355/gaiadr3',
    row_limit=-1
)

result = catalog.query_region(
    SkyCoord(ra=201.758127, dec=-47.3299358, unit=u.deg), 
    width=0.06 * u.deg,
    height=0.08 * u.deg
)[0]
result.write(data_dir + 'omega_Cen_GaiaDR3.ecsv')


catalog = Vizier(
    catalog='II/246/out',
    row_limit=-1
)
result = catalog.query_region(
    SkyCoord(ra=202, dec=-47, unit=u.deg), 
    width=0.8 * u.deg,
    height=0.8 * u.deg
)[0]

result.write(data_dir + 'omega_Cen_2MASS.ecsv')
```

</details>
<br /><br />

Load catalogs and launch mast-aladin:

In [ ]:
from astropy.table import Table
from sidecar import Sidecar

from mast_aladin import MastAladin
from mast_aladin.catalogs import RandomSubset, PriorityColumnSubset

# 0.2 deg radius cone search on TESS input catalog, n=92_174
tic = Table.read(data_dir / 'omega_Cen_TIC.ecsv')

# smaller rectangular region search on Gaia DR3, n=5_127
gaia_dr3 = Table.read(data_dir / 'omega_Cen_GaiaDR3.ecsv')

# larger rectangular region search on 2MASS, n=12_759
twoMASS = Table.read(data_dir / 'omega_Cen_2MASS.ecsv')

# 6 deg radius cone search on GALEX UV sources, n=13_081
galex = Table.read(data_dir / 'omega_Cen_galex.ecsv')

ma = MastAladin(full_screen=True, target='omega Cen', fov=1)

with Sidecar(anchor='split-right'):
    display(ma)

### ConvexHull

By default, `n_sources_max = 5_000`. If you call `add_table` on a table with `n_sources > n_sources_max`, a performance catalog layer will be constructed for you. The type of performance catalog (convex hull, random subset, prioritized subset) is set by `performance_cls` with the default `ConvexHull`.

ConvexHull displays up to `n_sources_max` sources as scatter points within the viewport. If `n_sources > n_sources_max`, the scatter points are replaced by a region overlay spanning the full area covered by the convex hull of the source coordinates for the full source catalog. This will be a polygon that contains every source.

In [ ]:
ma.add_table(
    table=tic,
    name='TIC',
    
    # marker settings:
    color='#ff0000',
    size=10,
    shape='cross',
)

You can set `n_sources_max` to a smaller number if you'd like even better performance:

In [ ]:
ma.add_table(
    table=gaia_dr3,
    name='Gaia DR3',
    n_sources_max=1000,
    
    # marker settings:
    color='#00ff00',
    size=20,
    shape='rhomb',
)

### RandomSubset

`RandomSubset` displays up to `n_sources_max` sources randomly drawn from within the viewport.

Note: a new random subset is selected for every update to the viewport. This means that interesting sources seen for one view may not be visible after pan/zoom. 

In [ ]:
twoMASS_perf_catalog = RandomSubset(
    twoMASS,
    name='2MASS',
    n_sources_max=1_000,
    
    # marker settings:
    color='#ffffff',
    size=20,
    shape='triangle',
)

ma.add_table(twoMASS_perf_catalog)

### PriorityColumnSubset

`PriorityColumnSubset` displays up to `n_sources_max` sources within the viewport, prioritizing sources with the smallest value in the `table` column named `priority_column_name`.

Set `small_value_high_priority = False` to prioritize the largest values with the highest priority. 

Note: compared with `RandomSubset`, sources displayed in one view are more likely to remain visible after pan/zoom. 

In [ ]:
galex_perf_catalog = PriorityColumnSubset(
    galex,
    name='GALEX',

    # viz config:
    n_sources_max=1_000,
    ra_column='RA_ICRS',
    dec_column='DE_ICRS',
    priority_column_name='Gmag',
    small_value_high_priority=True,

    # marker settings:
    size=10,
    shape='circle',
    color='#0090ff',
)
ma.add_table(galex_perf_catalog)

Inspect the performance catalogs that are in this instance of mast-aladin:

In [ ]:
ma.performance_catalogs

How many sources can be represented in this instance (without showing this many scatter points)?

In [ ]:
total_sources = sum([
    len(performance_catalog.table)
    for performance_catalog in ma.performance_catalogs
])

print(f"total sources represented in this visualization: {total_sources}")